# 05 · Tác động của từng bước làm sạch lên stay-point

**Câu hỏi:** mỗi bước làm sạch thay đổi stay-point bao nhiêu và ở đâu? Notebook này **không quyết định ngưỡng nào**: các quyết định đã có bằng chứng ở notebook 01–04 và 06–09. Đây là phần minh hoạ để kiểm chứng bằng mắt: chạy stay-point qua từng giai đoạn, so sánh, và vẽ **một bản đồ cho mỗi user** với mỗi giai đoạn là một lớp bật/tắt được.

| Giai đoạn | Dữ liệu | Cách chạy stay-point |
|---|---|---|
| **S0** | Thô: chỉ bỏ toạ độ ngoài phạm vi | **Nối tất cả file** của user (cách sai ban đầu) |
| **S1** | Thô | Cắt tại khoảng ngắt > 18 h, tức nối file liền nhau khi khoảng ngắt ≤ 18 h (`detect()` mặc định, notebook 04) |
| **S2** | + xử lý trùng timestamp (notebook 02) | Như S1 |
| **S3** | + loại điểm gai và kiểm tra lại nhóm có điểm neo là điểm gai, tức **dữ liệu sạch** | Như S1 |

Các bước còn lại của pipeline (altitude, segment, múi giờ) không đổi vị trí hay thời điểm của điểm GPS, nên không đổi stay-point. Bộ phát hiện: 200 m / 30 phút, CENTROID (notebook 07).

**So khớp stay-point giữa hai giai đoạn:** một stay-point được coi là **giữ nguyên** khi giai đoạn kia có stay-point với cùng giờ đến, giờ đi và tâm cách nhau dưới 1 m. Ngược lại, nó là **mới** (chỉ có ở giai đoạn sau) hoặc **mất** (chỉ có ở giai đoạn trước).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK = "05_staypoint_cleaning_impact"
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("outputs") / NOTEBOOK
FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR = (OUT_DIR / d for d in ("figures", "tables", "cache", "maps"))
for d in (FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(Path("../src").resolve()))
import json

import folium

from gps.data.processing import (
    DEFAULT_THRESHOLDS, _haversine_vectorized, filter_physical_bounds, load_plt_file,
    recheck_invalid_anchor_groups, remove_speed_spikes, resolve_duplicate_timestamps,
)
from gps.features.stay_point import DistanceMode, StayPointDetector

STAGES = ["S0", "S1", "S2", "S3"]
STAGE_LABEL = {"S0": "S0 raw, all files joined", "S1": "S1 raw, files chained <= 18 h",
               "S2": "S2 + duplicates resolved", "S3": "S3 + spikes removed (clean)"}
STAGE_COLOR = {"S0": "#7f7f7f", "S1": "#ff7f0e", "S2": "#9467bd", "S3": "#2ca02c"}
detector = StayPointDetector(30, 200, DistanceMode.CENTROID)
joined_detector = StayPointDetector(30, 200, DistanceMode.CENTROID, max_gap_hours=None)   # S0: nối tất cả, không cắt
STAY_CACHE = CACHE_DIR / "stays_by_stage"
STAY_CACHE.mkdir(exist_ok=True)


def user_stages(uid: str) -> dict[str, pd.DataFrame]:
    """Points of one user after each cleaning stage (per .plt file, as in clean_trajectory)."""
    raw, dedup, clean = [], [], []
    for path in sorted((DATA_DIR / uid / "Trajectory").glob("*.plt")):
        df = filter_physical_bounds(load_plt_file(path), DEFAULT_THRESHOLDS)
        if df.empty:
            continue
        df = df.assign(source_file=path.name)
        kept, q_dup = resolve_duplicate_timestamps(df, DEFAULT_THRESHOLDS)
        kept2, _ = remove_speed_spikes(kept, DEFAULT_THRESHOLDS)
        kept2, _ = recheck_invalid_anchor_groups(kept2, q_dup, DEFAULT_THRESHOLDS)
        raw.append(df)
        dedup.append(kept)
        clean.append(kept2)
    return {"raw": pd.concat(raw, ignore_index=True), "dedup": pd.concat(dedup, ignore_index=True),
            "clean": pd.concat(clean, ignore_index=True)}


def as_rows(stays, stage: str) -> list[dict]:
    return [{"stage": stage, "lat": sp.lat, "lon": sp.lon, "arrival": pd.Timestamp(sp.arrival_time),
             "departure": pd.Timestamp(sp.departure_time), "duration_min": sp.duration_minutes} for sp in stays]

## 1. Stay-point của 182 user qua 4 giai đoạn
Mỗi user chạy bộ phát hiện 4 lần trên khoảng 24.9 triệu điểm, nên lần đầu mất khoảng 1–2 giờ. Kết quả được cache theo từng user; chạy lại sẽ tiếp tục từ user chưa xong.

In [2]:
users = sorted(d.name for d in DATA_DIR.iterdir() if (d / "Trajectory").exists())
for k, uid in enumerate(users, start=1):
    path = STAY_CACHE / f"user_{uid}.parquet"
    if path.with_suffix(".json").exists():  # written last, so a user interrupted mid-way is redone
        continue
    print(f"\r[{k}/{len(users)}] user {uid}", end="", flush=True)
    st = user_stages(uid)
    rows = (as_rows(joined_detector.detect(st["raw"]), "S0") + as_rows(detector.detect(st["raw"]), "S1")
            + as_rows(detector.detect(st["dedup"]), "S2") + as_rows(detector.detect(st["clean"]), "S3"))
    pd.DataFrame(rows, columns=["stage", "lat", "lon", "arrival", "departure", "duration_min"]).assign(
        user_id=uid).to_parquet(path)
    path.with_suffix(".json").write_text(json.dumps({k: len(v) for k, v in st.items()}))
print()
# Point counts live in a separate file: a user without any stay-point has no rows to carry them.
stays = pd.concat([d for p in sorted(STAY_CACHE.glob("user_*.parquet")) if len(d := pd.read_parquet(p))], ignore_index=True)
points = pd.DataFrame({p.stem[5:]: json.loads(p.read_text()) for p in sorted(STAY_CACHE.glob("user_*.json"))}).T
no_stay = points.index.difference(stays["user_id"].unique())
print(f"Users without any stay-point at any stage: {len(no_stay)} of {len(points)} {list(no_stay)} "
      f"({points.loc[no_stay, 'clean'].sum():,} clean points)")
processed = sum(pd.read_parquet(p, columns=["lat"]).shape[0] for p in Path("../data/processed/users").glob("user_*.parquet"))
print(f"Points: raw {points['raw'].sum():,} -> dedup {points['dedup'].sum():,} -> clean {points['clean'].sum():,} "
      f"(pipeline output: {processed:,})")

Users without any stay-point at any stage: 7 of 182 ['049', '072', '116', '137', '149', '170', '178'] (32,363 clean points)


Points: raw 24,876,977 -> dedup 24,178,013 -> clean 24,177,459 (pipeline output: 24,177,459)


## 2. Tổng quan theo giai đoạn

In [3]:
by_stage = stays.groupby("stage").agg(stay_points=("lat", "size"), users=("user_id", "nunique"),
                                      dwell_hours=("duration_min", lambda d: d.sum() / 60),
                                      longest_h=("duration_min", lambda d: d.max() / 60),
                                      over_48h=("duration_min", lambda d: int((d > 48 * 60).sum())))
by_stage.to_csv(TABLE_DIR / "staypoints_by_stage.csv")
by_stage.round(1)

,stay_points,users,dwell_hours,longest_h,over_48h
stage,,,,,
S0,19443,175,203070.4,35023.6,282
S1,18598,174,79389.0,43.8,0
S2,18597,174,79376.6,43.8,0
S3,18597,174,79376.6,43.8,0


## 3. Mỗi bước thay đổi gì?
`unchanged`: stay-point giống hệt ở cả hai giai đoạn. `added` / `removed`: chỉ có ở giai đoạn sau / trước. `users_changed`: số user có ít nhất một stay-point thay đổi.

In [4]:
def stay_key(df: pd.DataFrame) -> pd.Series:
    return (df["user_id"] + "|" + df["arrival"].astype(str) + "|" + df["departure"].astype(str)
            + "|" + df["lat"].round(5).astype(str) + "|" + df["lon"].round(5).astype(str))


stays["key"] = stay_key(stays)
rows, per_user = [], []
for a, b in zip(STAGES[:-1], STAGES[1:]):
    ka, kb = set(stays.loc[stays["stage"] == a, "key"]), set(stays.loc[stays["stage"] == b, "key"])
    changed_users = {k.split("|")[0] for k in ka ^ kb}
    rows.append({"step": f"{a} -> {b}", "unchanged": len(ka & kb), "removed": len(ka - kb), "added": len(kb - ka),
                 "users_changed": len(changed_users)})
    per_user.append(pd.Series({u: sum(1 for k in ka ^ kb if k.startswith(u + "|")) for u in changed_users}, name=f"{a}->{b}"))
steps = pd.DataFrame(rows)
steps.to_csv(TABLE_DIR / "staypoint_changes_by_step.csv", index=False)
user_changes = pd.concat(per_user, axis=1).fillna(0).astype(int).sort_index()
user_changes.to_csv(TABLE_DIR / "staypoint_changes_by_user.csv")
steps

,step,unchanged,removed,added,users_changed
0,S0 -> S1,18405,1038,193,137
1,S1 -> S2,18446,152,151,31
2,S2 -> S3,18597,0,0,0


### 3.1 S1 → S2: xử lý trùng làm stay-point đổi đến đâu?
Ghép mỗi stay-point **mất** ở S1 với các stay-point **mới** ở S2 của cùng user mà chồng thời gian với nó, rồi đo độ lệch tâm (tới stay-point mới gần nhất) và thay đổi thời lượng. Cột `quarantine_inside` là lý do của các điểm bị quarantine trong khoảng thời gian bị ảnh hưởng (hợp của các khoảng S1 và S2 liên quan).

**Thay đổi nhỏ:** khớp 1:1, tâm lệch ≤ 20 m, thời lượng lệch ≤ 5 phút. Còn lại là **thay đổi lớn**.

In [5]:
quarantine = pd.concat([pd.read_csv(p, parse_dates=["datetime"]).assign(user_id=p.stem[5:])
                        for p in sorted(Path("../data/processed/quarantine").glob("user_*.csv"))], ignore_index=True)
s1_keys, s2_keys = (set(stays.loc[stays["stage"] == s, "key"]) for s in ("S1", "S2"))
lost = stays[(stays["stage"] == "S1") & ~stays["key"].isin(s2_keys)]
new = stays[(stays["stage"] == "S2") & ~stays["key"].isin(s1_keys)]


def overlapping(df: pd.DataFrame, r) -> pd.DataFrame:
    return df[(df["user_id"] == r.user_id) & (df["arrival"] < r.departure) & (df["departure"] > r.arrival)]


pairs = []
for r in lost.itertuples():
    c = overlapping(new, r)
    d = _haversine_vectorized(np.full(len(c), r.lat), np.full(len(c), r.lon), c["lat"].to_numpy(), c["lon"].to_numpy())
    m = c.iloc[d.argmin()] if len(c) else None
    lo, hi = min([r.arrival, *c["arrival"]]), max([r.departure, *c["departure"]])
    q = quarantine[(quarantine["user_id"] == r.user_id) & quarantine["datetime"].between(lo, hi)]
    pairs.append({"user_id": r.user_id, "arrival": r.arrival, "duration_min": r.duration_min, "overlapping_s2": len(c),
                  "shift_m": d.min() if len(c) else np.nan,
                  "duration_change_min": m["duration_min"] - r.duration_min if len(c) else np.nan,
                  "same_times": len(c) == 1 and m["arrival"] == r.arrival and m["departure"] == r.departure,
                  "quarantine_inside": ", ".join(sorted(q["reason"].unique()))})
pairs = pd.DataFrame(pairs)
small = (pairs["overlapping_s2"] == 1) & (pairs["shift_m"] <= 20) & (pairs["duration_change_min"].abs() <= 5)
pairs["kind"] = np.where(small, "small", "large")
pairs.to_csv(TABLE_DIR / "s1_s2_changes.csv", index=False)
sm = pairs[small]
print(f"Lost S1 stays: {len(lost)}, new S2 stays: {len(new)}, users: {pairs['user_id'].nunique()}")
print(f"Small: {len(sm)} | centroid shift median {sm['shift_m'].median():.1f} m, P90 {sm['shift_m'].quantile(0.9):.1f} m | "
      f"same arrival and departure: {int(sm['same_times'].sum())} | |duration change| P90 {sm['duration_change_min'].abs().quantile(0.9):.1f} min")
print("Dwell S1 -> S2:", f"{(new['duration_min'].sum() - lost['duration_min'].sum()) / 60:+.1f} h")
only_new = [r for r in new.itertuples() if overlapping(lost, r).empty]
for r in only_new:
    q = quarantine[(quarantine["user_id"] == r.user_id) & quarantine["datetime"].between(r.arrival, r.departure)]
    print(f"New S2 stay with no overlapping S1 stay: user {r.user_id} {r.arrival}, {r.duration_min:.0f} min, "
          f"quarantined inside: {q['reason'].value_counts().to_dict()}")
pairs[~small].round({"duration_min": 1, "shift_m": 1, "duration_change_min": 1})

Lost S1 stays: 152, new S2 stays: 151, users: 29
Small: 136 | centroid shift median 1.3 m, P90 4.4 m | same arrival and departure: 50 | |duration change| P90 0.1 min
Dwell S1 -> S2: -12.4 h
New S2 stay with no overlapping S1 stay: user 009 2008-11-08 10:41:38, 73 min, quarantined inside: {'duplicate_ambiguous_candidates': 2}
New S2 stay with no overlapping S1 stay: user 128 2008-05-18 00:00:16, 32 min, quarantined inside: {'duplicate_ambiguous_candidates': 5}


,user_id,arrival,duration_min,overlapping_s2,shift_m,duration_change_min,same_times,quarantine_inside,kind
11,013,2008-10-28 09:26:43,51.9,1,26.4,722.2,False,duplicate_ambiguous_candidates,large
12,013,2008-10-28 10:18:40,722.6,2,51.8,-246.4,False,duplicate_ambiguous_candidates,large
14,013,2008-11-16 05:49:43,46.0,1,66.6,13.8,False,duplicate_ambiguous_candidates,large
19,030,2009-05-23 07:04:08,58.7,1,31.2,28.0,False,duplicate_ambiguous_candidates,large
22,058,2011-08-29 11:18:06,33.5,2,1.3,-0.0,False,,large
25,059,2011-08-29 11:18:06,33.5,2,1.3,-0.0,False,,large
28,065,2011-08-25 00:58:25,828.5,0,NaN,NaN,False,,large
70,119,2008-09-24 02:43:51,75.7,0,NaN,NaN,False,duplicate_ambiguous_candidates,large
71,119,2008-09-24 03:59:35,39.9,1,0.5,-5.8,False,duplicate_ambiguous_candidates,large
89,141,2011-08-29 11:18:06,33.5,2,1.4,-0.0,False,,large


Ví dụ một thay đổi lớn **không có điểm bị quarantine**: user 065, ngày 2011-08-25. Ở S1, stay-point 13.8 h bắt đầu từ điểm cuối của một file và kéo sang file sau, sau khoảng ngắt 13.8 h. Bỏ các bản sao trùng làm tâm cửa sổ xê dịch vài mét, và khoảng cách sang file sau vượt 200 m.

In [6]:
files = ["20110825003603.plt", "20110825144457.plt"]
ex_raw = pd.concat([filter_physical_bounds(load_plt_file(DATA_DIR / "065" / "Trajectory" / f), DEFAULT_THRESHOLDS).assign(source_file=f)
                    for f in files], ignore_index=True)
ex_dedup = pd.concat([resolve_duplicate_timestamps(g, DEFAULT_THRESHOLDS)[0] for _, g in ex_raw.groupby("source_file")], ignore_index=True)
for name, d in (("S1 raw", ex_raw), ("S2 dedup", ex_dedup)):
    d = d.sort_values("datetime", kind="stable")
    a, b = d[d["source_file"] == files[0]].iloc[-1], d[d["source_file"] == files[1]].iloc[0]
    gap_m = _haversine_vectorized(np.array([a["lat"]]), np.array([a["lon"]]), np.array([b["lat"]]), np.array([b["lon"]]))[0]
    found = [f"{sp.arrival_time:%H:%M} -> {sp.departure_time:%H:%M}, duration {sp.duration_minutes:.0f} min, observed {sp.observed_minutes:.1f} min"
             for sp in detector.detect(d)]
    print(f"{name}: {len(d):,} points, last -> next file {gap_m:.2f} m | stays: {found or 'none'}")

S1 raw: 6,899 points, last -> next file 200.00 m | stays: ['00:58 -> 14:46, duration 829 min, observed 2.0 min']
S2 dedup: 1,439 points, last -> next file 200.54 m | stays: none


### 3.2 S2 → S3: vì sao loại điểm gai không đổi stay-point nào?

In [7]:
spikes = quarantine[quarantine["reason"] == "speed_spike"]
s3 = stays[stays["stage"] == "S3"]
margin = pd.Timedelta("30min")
near_stay = spikes.apply(lambda q: ((s3["user_id"] == q["user_id"]) & (s3["arrival"] - margin <= q["datetime"])
                                    & (s3["departure"] + margin >= q["datetime"])).any(), axis=1)
print("Spike points by user:", spikes["user_id"].value_counts().to_dict())
print(f"Spike points inside or within 30 min of a stay-point: {int(near_stay.sum())} of {len(spikes)}")

Spike points by user: {'062': 564, '042': 1, '128': 1, '144': 1}
Spike points inside or within 30 min of a stay-point: 0 of 567


## 4. Ví dụ S0: stay-point dài bất thường khi nối tất cả file
Các stay-point dài nhất ở S0. Chúng được ghép từ những lần ghi cách nhau nhiều ngày, thậm chí nhiều năm, và biến mất ở S1.

In [8]:
s0 = stays[stays["stage"] == "S0"].nlargest(8, "duration_min")
s0.assign(duration_days=(s0["duration_min"] / 1440).round(1))[["user_id", "arrival", "departure", "duration_days"]]

,user_id,arrival,departure,duration_days
37693,055,2007-08-09 01:35:41,2011-08-07 09:11:01,1459.3
56967,140,2007-05-09 02:20:00,2008-08-18 13:35:07,467.5
43606,082,2007-06-26 01:29:09,2007-11-28 12:31:27,155.5
52120,128,2007-07-30 02:04:22,2007-11-28 12:31:00,121.4
68995,163,2007-07-20 01:46:16,2007-10-09 13:51:50,81.5
57934,142,2010-07-06 01:56:00,2010-09-20 10:50:56,76.4
36825,052,2008-03-19 07:36:23,2008-05-20 09:40:57,62.1
58680,144,2009-03-26 13:30:30,2009-05-20 10:23:48,54.9


## 5. Bản đồ cho từng user
Chọn theo quy tắc cố định:
- 3 user có **nhiều stay-point thay đổi nhất** do làm sạch (S1 → S3);
- 1 user **điển hình** có thay đổi (trung vị của các user có thay đổi);
- user có stay-point S0 dài nhất.

Trên mỗi bản đồ, mỗi giai đoạn là một lớp vòng tròn 200 m (bấm để xem giờ đến và thời lượng). Lớp **"Changed by cleaning"** chỉ hiện các stay-point khác nhau giữa S1 và S3: đỏ là mất do làm sạch, xanh dương là mới có sau làm sạch. Lớp **"Quarantined points"** là các điểm bị loại, kèm lý do. Lớp quỹ đạo sạch (đã thưa bớt điểm) mặc định tắt.

In [9]:
s1s3 = user_changes.get("S1->S2", 0) + user_changes.get("S2->S3", 0)
changed = s1s3[s1s3 > 0].sort_index().sort_values(kind="stable")  # ties broken by user id
picks = list(changed.index[-3:][::-1]) + [changed.index[len(changed) // 2]] + [stays.loc[stays["stage"] == "S0"].nlargest(1, "duration_min")["user_id"].iloc[0]]
picks = list(dict.fromkeys(picks))
quarantine_dir = Path("../data/processed/quarantine")
for uid in picks:
    u = stays[stays["user_id"] == uid]
    m = folium.Map(location=[u["lat"].median(), u["lon"].median()], zoom_start=12)
    for stage in STAGES:
        layer = folium.FeatureGroup(name=f"{STAGE_LABEL[stage]} ({int((u['stage'] == stage).sum())})", show=stage in ("S1", "S3"))
        for r in u[u["stage"] == stage].itertuples():
            folium.Circle([r.lat, r.lon], radius=200, color=STAGE_COLOR[stage], weight=2, fill=False,
                          popup=f"{stage}: {r.arrival:%Y-%m-%d %H:%M} GMT, {r.duration_min / 60:.1f} h").add_to(layer)
        layer.add_to(m)
    k1, k3 = set(u.loc[u["stage"] == "S1", "key"]), set(u.loc[u["stage"] == "S3", "key"])
    diff = folium.FeatureGroup(name=f"Changed by cleaning S1 -> S3 (removed {len(k1 - k3)}, added {len(k3 - k1)})")
    for r in u[(u["stage"].isin(["S1", "S3"])) & u["key"].isin(k1 ^ k3)].itertuples():
        folium.Circle([r.lat, r.lon], radius=200, color="#d62728" if r.stage == "S1" else "#1f77b4", weight=4, fill=True,
                      fill_opacity=0.2, popup=f"{'removed' if r.stage == 'S1' else 'added'}: {r.arrival:%Y-%m-%d %H:%M} GMT, "
                                              f"{r.duration_min / 60:.1f} h").add_to(diff)
    diff.add_to(m)
    qpath = quarantine_dir / f"user_{uid}.csv"
    if qpath.exists():
        q = pd.read_csv(qpath)
        ql = folium.FeatureGroup(name=f"Quarantined points ({len(q)})")
        for r in q.itertuples():
            folium.CircleMarker([r.lat, r.lon], radius=3, color="black", fill=True, tooltip=f"{r.reason} {r.datetime}").add_to(ql)
        ql.add_to(m)
    clean = pd.read_parquet(f"../data/processed/users/user_{uid}.parquet", columns=["source_file", "datetime", "lat", "lon"])
    tl = folium.FeatureGroup(name="Clean trajectory (thinned)", show=False)
    step = max(1, len(clean) // 4000)
    for _, f in clean.sort_values("datetime").groupby("source_file"):
        pts = f[["lat", "lon"]].to_numpy()[::step]
        if len(pts) >= 2:
            folium.PolyLine(pts, color="#555555", weight=1, opacity=0.6).add_to(tl)
    tl.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    out = MAP_DIR / f"staypoint_stages_user{uid}.html"
    m.save(out)
    print(f"user {uid}: {out} ({out.stat().st_size / 1e6:.1f} MB)")

user 065: outputs\05_staypoint_cleaning_impact\maps\staypoint_stages_user065.html (0.6 MB)


user 167: outputs\05_staypoint_cleaning_impact\maps\staypoint_stages_user167.html (2.0 MB)


user 141: outputs\05_staypoint_cleaning_impact\maps\staypoint_stages_user141.html (0.3 MB)


user 096: outputs\05_staypoint_cleaning_impact\maps\staypoint_stages_user096.html (0.6 MB)
user 055: outputs\05_staypoint_cleaning_impact\maps\staypoint_stages_user055.html (0.3 MB)


## Kết luận

| Bước | Điểm GPS | Stay-point | Ý nghĩa |
|---|---|---|---|
| **S0 → S1**: nối file ≤ 18 h thay vì nối tất cả | Không đổi (24,876,977) | 19,443 → 18,598. Mất 1,038, mới 193, ở **137 user**. Tổng thời gian ở lại giảm từ 203,070 h xuống 79,389 h. Stay-point dài nhất giảm từ 1,459 ngày (user 055) xuống 43.8 h. Stay-point > 48 h giảm từ 282 xuống 0 | **Bước tác động lớn nhất.** Nối tất cả file tạo stay-point giả từ các lần ghi cách nhau nhiều ngày, thậm chí nhiều năm |
| **S1 → S2**: xử lý trùng timestamp | 24,876,977 → 24,178,013 (−2.8%) | Mất 152, mới 151, ở 31 user (0.8% số stay-point). Tổng thời gian ở lại giảm 12.4 h (−0.016%). 136/152 là thay đổi nhỏ: tâm lệch trung vị 1.3 m (P90 4.4 m), và 50 stay-point giữ nguyên giờ đến và giờ đi | Gần như không đổi |
| **S2 → S3**: loại điểm gai và kiểm tra lại nhóm có điểm neo là điểm gai | 24,178,013 → 24,177,459, khớp đúng đầu ra pipeline | **0 thay đổi** | 564/567 điểm gai thuộc user 062. Không điểm gai nào nằm trong hoặc cách stay-point dưới 30 phút: điểm gai nằm trong đoạn di chuyển |

**Thay đổi lớn ở S1 → S2** gồm 16 stay-point mất và 2 stay-point chỉ có ở S2:
- **10 trường hợp, cộng cả 2 stay-point chỉ có ở S2, nằm cạnh một nhóm `duplicate_ambiguous_candidates`.** Ở dữ liệu thô, ứng viên lệch xa làm cửa sổ bị cắt ngang. Khi cả nhóm bị quarantine, cửa sổ nối liền lại. Ví dụ user 013: hai stay-point 0.9 h và 12.0 h ở S1 thành một stay-point 12.9 h ở S2. Quarantine nhóm mơ hồ (phương án A, notebook 02) **không làm mất chỗ ở nào**: nó gỡ điểm sai ra khỏi chỗ ở.
- **6 trường hợp không có điểm nào bị quarantine.** Nguyên nhân là việc gộp các bản sao trùng toạ độ. Ở chế độ CENTROID, tâm là trung bình theo điểm, nên một vị trí có 5 bản sao nặng gấp 5 lần. Bỏ bản sao làm tâm xê dịch vài mét, và cửa sổ đang sát ngưỡng 200 m thì đổi kết quả. Ví dụ user 065: khoảng cách sang file sau là 200.00 m ở S1 và 200.54 m ở S2. Stay-point 829 phút ở S1 thực ra chỉ có **2.0 phút** có điểm GPS. S2 đúng hơn, vì mỗi thời điểm chỉ còn một vị trí.

**Quyết định: không thay đổi `src/`.** Notebook xác nhận ba điều:
1. Nối file theo khoảng ngắt ≤ 18 h (notebook 04, đã áp dụng) là thay đổi quyết định chất lượng stay-point.
2. Xử lý trùng và loại điểm gai hầu như không đổi stay-point. Giá trị của chúng nằm ở quỹ đạo sạch và ở độ đúng của từng điểm (segment, bản đồ, đầu vào API), không nằm ở số lượng stay-point.
3. Không bước làm sạch nào xoá một chỗ ở thật. Các stay-point bị mất hoặc chỉ là biến thể lệch vài mét, hoặc là stay-point sát ngưỡng và gần như không có điểm GPS.

**Giới hạn và phát hiện phụ:**
- **Stay-point sát ngưỡng nhạy với vài mét.** Đây là bản chất của ngưỡng cứng 200 m. `observed_minutes` giúp nhận ra stay-point "rỗng" (ví dụ 2.0/829 phút). Phát hiện này liên quan đến hai việc sắp làm: cắt đuôi cửa sổ CENTROID và chia chuỗi tại mọi khoảng ngắt > 18 h.
- **7/182 user không có stay-point nào ở mọi giai đoạn** (049, 072, 116, 137, 149, 170, 178; tổng cộng 32,363 điểm). Classifier phải trả kết quả "unknown" cho họ (mục 3.5 trong lộ trình).
- **Một số file `.plt` giống hệt nhau từng byte giữa các user.** User 058, 059, 141 có chung stay-point ngày 2011-08-29, user 153 và 163 có chung stay-point ngày 2009-04-19, vì chung file gốc. Giống hệt từng byte nghĩa là **cùng một log của một máy**: hai máy đi cùng nhau vẫn khác nhau do nhiễu GPS và giờ ghi. Lý do file bị chép rất có thể là họ đi cùng nhau, và khi đó chuyến đi đúng là của cả hai. Cần một notebook riêng để kiểm tra điều này trước khi đặt quy tắc nào.
- Giờ trên popup bản đồ là GMT.